# Understanding $N_{\text{eff}}$ and the Hybrid Correlation Blend

## Context: Why do we need $\alpha$?

The pipeline produces **energy-correlated Legendre coefficient samples** via a covariance matrix.
Two methods estimate the **correlation structure** between energy bins:

| Method | How it works | Strength | Weakness |
|--------|-------------|----------|----------|
| **Gaussian parametric** | Fits a Gaussian decay $\rho(E_i, E_j) = e^{-(E_i - E_j)^2 / 2\ell^2}$ to per-bin MC samples | Smooth, always available | Assumes a parametric shape — may miss real structure |
| **Kernel-weight MC (KW)** | Perturbs shared EXFOR datasets across bins → correlations emerge naturally from overlapping data | Data-driven, captures real structure | Noisy when few experiments overlap a bin |

The **hybrid method** blends them per energy bin:

$$\rho_{\text{hybrid}}(i,j) = \alpha_{ij} \cdot \rho_{\text{KW}}(i,j) + (1 - \alpha_{ij}) \cdot \rho_{\text{Gauss}}(i,j)$$

where $\alpha_{ij} = \min(\alpha_i, \alpha_j)$ (conservative: the weakest bin dictates).

**The question**: How do we decide $\alpha_i$ for each bin? → That's where $N_{\text{eff,kw}}$ comes in.

## Step 1: Overlap weights — which experiments "see" each bin?

Each ENDF energy bin has boundaries $[E_{\text{low}}, E_{\text{high}}]$ and an **energy resolution** $\sigma_E$ (from TOF parameters — it is a property of the bin, not of the experiment).

Each EXFOR experiment (identified by `entry.subentry`) measures angular distributions at one or more energies $E_j$. The **overlap weight** for experiment $k$ at a given bin is the probability that its measurement energy falls within the bin, assuming the bin's resolution:

$$w_k = \Phi\!\left(\frac{E_{\text{high}} - E_k}{\sigma_E}\right) - \Phi\!\left(\frac{E_{\text{low}} - E_k}{\sigma_E}\right)$$

where $\Phi$ is the standard normal CDF and $\sigma_E$ is the **bin's** energy resolution (`bin_info.sigma_E_mev`). This is computed in `compute_overlap_weight()`.

**Key detail — deduplication per experiment:** If the same experiment (`entry.subentry`) measured at multiple energies (e.g. 4.9, 5.0, 5.1 MeV), `precompute_overlap_weights()` keeps only the energy with the **highest overlap weight** for that experiment. This prevents experiments with dense energy grids from counting multiple times. The number of angular data points in the experiment is irrelevant — only energy proximity to the bin matters.

- Experiment energy well inside the bin → $w \approx 1$
- Experiment energy far outside the bin → $w \approx 0$
- Wide $\sigma_E$ (poor resolution) → weight smears across multiple bins

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- Visualise overlap weights for a single bin ---
bin_low, bin_high = 4.8, 5.2  # ENDF bin [4.8, 5.2] MeV
sigma_E = 0.15                # Bin's energy resolution (from TOF parameters)

# Three experiments at different energies (sigma_E is the SAME for all — it's the bin's property)
experiments = [
    {"name": "Exp A (centered in bin)",  "E": 5.0,  "color": "C0"},
    {"name": "Exp B (near bin edge)",    "E": 5.35, "color": "C1"},
    {"name": "Exp C (far from bin)",     "E": 6.0,  "color": "C2"},
]

fig, ax = plt.subplots(figsize=(9, 4))
E_range = np.linspace(3.5, 7.0, 500)

# Shade the bin
ax.axvspan(bin_low, bin_high, alpha=0.15, color="grey", label=f"ENDF bin [{bin_low}, {bin_high}] MeV")

for exp in experiments:
    # The Gaussian is centred on the experiment's energy, with the BIN's sigma_E
    pdf = norm.pdf(E_range, exp["E"], sigma_E)
    w = norm.cdf(bin_high, exp["E"], sigma_E) - norm.cdf(bin_low, exp["E"], sigma_E)
    ax.plot(E_range, pdf, color=exp["color"], lw=2,
            label=f'{exp["name"]}  (E={exp["E"]:.2f} MeV):  w = {w:.3f}')
    ax.fill_between(E_range, pdf, where=(E_range >= bin_low) & (E_range <= bin_high),
                     alpha=0.25, color=exp["color"])

ax.set_xlabel("Energy (MeV)", fontsize=12)
ax.set_ylabel("Probability density", fontsize=12)
ax.set_title(f"Overlap weight = shaded area   (all use the bin's $\\sigma_E$ = {sigma_E} MeV)", fontsize=13)
ax.legend(fontsize=10, loc="upper right")
ax.set_xlim(3.5, 7.0)
plt.tight_layout()
plt.show()

## Step 2: $N_{\text{eff,kw}}$ — Kish's effective sample size at the experiment level

Once we have the overlap weights $w_k$ for each experiment $k$ at a given bin, we normalize them to fractions and compute the **effective number of experiments**:

$$f_k = \frac{w_k}{\sum_k w_k}, \qquad N_{\text{eff,kw}} = \frac{1}{\sum_k f_k^2}$$

This is [Kish's effective sample size](https://en.wikipedia.org/wiki/Effective_sample_size). It answers: **"How many equally-weighted experiments is this equivalent to?"**

| Scenario | $N_{\text{eff,kw}}$ |
|----------|---------------------|
| 5 experiments, all equal weight | 5.0 |
| 5 experiments, one dominates (90%) | ≈ 1.2 |
| 2 experiments, equal weight | 2.0 |
| 1 experiment | 1.0 |

This is computed in `compute_kw_diagnostics()` (`exfor_utils.py:300`).

In [ ]:
# --- Demonstrate N_eff for different weight distributions ---

def kish_n_eff(weights):
    """Kish's effective sample size."""
    w = np.asarray(weights, dtype=float)
    f = w / w.sum()
    return 1.0 / np.sum(f**2)

scenarios = {
    "5 equal experiments":          [1.0, 1.0, 1.0, 1.0, 1.0],
    "5 exp, one dominates (90%)":   [0.9, 0.025, 0.025, 0.025, 0.025],
    "3 equal experiments":          [1.0, 1.0, 1.0],
    "3 exp, one dominates (70%)":   [0.7, 0.15, 0.15],
    "2 equal experiments":          [1.0, 1.0],
    "2 exp, one dominates (85%)":   [0.85, 0.15],
    "1 experiment":                 [1.0],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.3, 1]})

# Left: bar chart of weight distributions
ax = axes[0]
y_pos = 0
y_ticks, y_labels = [], []
colors_map = plt.cm.Set2(np.linspace(0, 1, 8))

for i, (name, weights) in enumerate(scenarios.items()):
    n_eff = kish_n_eff(weights)
    fracs = np.array(weights) / np.sum(weights)
    for j, f in enumerate(fracs):
        ax.barh(y_pos, f, height=0.7, color=colors_map[i], edgecolor="white", lw=0.5)
        if f > 0.08:
            ax.text(f/2, y_pos, f"{f:.0%}", ha="center", va="center", fontsize=8)
        y_pos += 1
    y_ticks.append(y_pos - len(weights)/2)
    y_labels.append(f"{name}\n$N_{{\\text{{eff}}}}$ = {n_eff:.2f}")
    y_pos += 1  # gap between scenarios

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel("Weight fraction $f_k$", fontsize=11)
ax.set_title("Experiment weight distributions", fontsize=12)
ax.set_xlim(0, 1.05)
ax.invert_yaxis()

# Right: N_eff as a function of weight concentration
ax2 = axes[1]
# For N experiments, vary the dominant fraction
for n_exp in [2, 3, 5, 8]:
    dominant_fracs = np.linspace(1/n_exp, 0.99, 200)
    n_effs = []
    for d in dominant_fracs:
        rest = (1 - d) / (n_exp - 1) if n_exp > 1 else 0
        w = [d] + [rest] * (n_exp - 1)
        n_effs.append(kish_n_eff(w))
    ax2.plot(dominant_fracs, n_effs, lw=2, label=f"$n_{{\\text{{exp}}}}$ = {n_exp}")

ax2.set_xlabel("Dominant experiment fraction $f_{\\max}$", fontsize=11)
ax2.set_ylabel("$N_{\\text{eff,kw}}$", fontsize=11)
ax2.set_title("$N_{\\text{eff}}$ drops as one experiment dominates", fontsize=12)
ax2.legend(fontsize=10)
ax2.set_ylim(0, 9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: From $N_{\text{eff,kw}}$ to $\alpha$ — the sigmoid + penalties

The function `compute_kw_reliability_alpha()` (`exfor_utils.py:310`) maps $N_{\text{eff,kw}}$ to a blend weight $\alpha \in [\alpha_{\min}, \alpha_{\max}]$ via three stages:

### 3a. Base sigmoid

$$\alpha_{\text{base}} = \alpha_{\min} + \frac{\alpha_{\max} - \alpha_{\min}}{1 + \exp\!\left(-\frac{N_{\text{eff}} - N_{\text{mid}}}{N_{\text{scale}}}\right)}$$

With defaults: $\alpha_{\min}=0.05$, $\alpha_{\max}=0.85$, $N_{\text{mid}}=3.5$, $N_{\text{scale}}=1.5$.

This means:
- $N_{\text{eff}} \ll 3.5$ → $\alpha \approx 0.05$ (almost pure Gaussian)
- $N_{\text{eff}} = 3.5$ → $\alpha \approx 0.45$ (half-and-half)
- $N_{\text{eff}} \gg 3.5$ → $\alpha \approx 0.85$ (mostly KW)

### 3b. Experiment count penalty

If fewer than `min_experiments=2` experiments contribute:

$$\alpha \leftarrow \alpha \times (0.25 + 0.25 \times n_{\text{exp}})$$

So 1 experiment → multiply by 0.5, 0 experiments → multiply by 0.25.

### 3c. Dominance penalty

If the largest experiment contributes more than `dominance_threshold=0.40`:

$$\alpha \leftarrow \alpha \times \left(1 - 0.5 \cdot \frac{f_{\max} - 0.40}{1 - 0.40}\right)$$

This penalizes bins where one experiment overwhelms the others (even if $N_{\text{eff}}$ looks OK due to many tiny contributions).

Final result is clamped to $[\alpha_{\min}, \alpha_{\max}] = [0.05, 0.85]$.

In [ ]:
# --- Reproduce compute_kw_reliability_alpha and visualize each stage ---

def sigmoid_alpha(n_eff, alpha_min=0.05, alpha_max=0.85, n_eff_mid=3.5, n_eff_scale=1.5):
    """Base sigmoid (no penalties)."""
    x = (n_eff - n_eff_mid) / n_eff_scale
    return alpha_min + (alpha_max - alpha_min) / (1.0 + np.exp(-x))

def full_alpha(n_eff, n_experiments, f_max,
               alpha_min=0.05, alpha_max=0.85, n_eff_mid=3.5, n_eff_scale=1.5,
               min_experiments=2, dominance_threshold=0.40):
    """Full alpha with experiment count + dominance penalties."""
    alpha = sigmoid_alpha(n_eff, alpha_min, alpha_max, n_eff_mid, n_eff_scale)
    # Experiment count penalty
    if n_experiments < min_experiments:
        alpha *= 0.25 + 0.25 * n_experiments
    # Dominance penalty
    if f_max > dominance_threshold:
        alpha *= 1.0 - 0.5 * (f_max - dominance_threshold) / (1.0 - dominance_threshold)
    return np.clip(alpha, alpha_min, alpha_max)

# --- Plot 1: Base sigmoid ---
n_eff_range = np.linspace(0, 10, 300)
alpha_base = sigmoid_alpha(n_eff_range)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.plot(n_eff_range, alpha_base, "C0", lw=2.5)
ax.axhline(0.05, color="grey", ls="--", lw=1, label="$\\alpha_{\\min}=0.05$")
ax.axhline(0.85, color="grey", ls=":", lw=1, label="$\\alpha_{\\max}=0.85$")
ax.axvline(3.5, color="C3", ls="--", lw=1, alpha=0.7, label="$N_{\\text{mid}}=3.5$")
ax.fill_between(n_eff_range, 0, alpha_base, alpha=0.1, color="C0")
ax.set_xlabel("$N_{\\text{eff,kw}}$", fontsize=12)
ax.set_ylabel("$\\alpha$ (base sigmoid)", fontsize=12)
ax.set_title("3a. Base sigmoid", fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
# Annotate key regions
ax.annotate("≈ pure Gaussian", xy=(1.0, 0.12), fontsize=9, color="C3", ha="center")
ax.annotate("≈ pure KW", xy=(8, 0.78), fontsize=9, color="C0", ha="center")

# --- Plot 2: Effect of experiment count penalty ---
ax = axes[1]
for n_exp in [1, 2, 3, 5]:
    alphas = []
    for ne in n_eff_range:
        # f_max = 1/n_exp (equal weights, no dominance penalty)
        alphas.append(full_alpha(ne, n_experiments=n_exp, f_max=1.0/max(n_exp,1)))
    ax.plot(n_eff_range, alphas, lw=2, label=f"$n_{{\\text{{exp}}}}$ = {n_exp}")
ax.set_xlabel("$N_{\\text{eff,kw}}$", fontsize=12)
ax.set_ylabel("$\\alpha$ (with exp. count penalty)", fontsize=12)
ax.set_title("3b. Experiment count penalty", fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# --- Plot 3: Effect of dominance penalty ---
ax = axes[2]
f_max_range = np.linspace(0.2, 0.95, 200)
for n_eff_val in [2, 3.5, 5, 8]:
    alphas = [full_alpha(n_eff_val, n_experiments=5, f_max=f) for f in f_max_range]
    ax.plot(f_max_range, alphas, lw=2, label=f"$N_{{\\text{{eff}}}}$ = {n_eff_val}")
ax.axvline(0.40, color="C3", ls="--", lw=1, alpha=0.7, label="threshold = 0.40")
ax.set_xlabel("$f_{\\max}$ (dominant exp. fraction)", fontsize=12)
ax.set_ylabel("$\\alpha$", fontsize=12)
ax.set_title("3c. Dominance penalty ($n_{\\text{exp}}$=5)", fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: How $\alpha$ affects the final correlation matrix

For each pair of energy bins $(i, j)$, the hybrid correlation is:

$$\rho_{\text{hybrid}}(i,j) = \alpha_{ij} \cdot \rho_{\text{KW}}(i,j) + (1 - \alpha_{ij}) \cdot \rho_{\text{Gauss}}(i,j)$$

where $\alpha_{ij} = \min(\alpha_i, \alpha_j)$ — the **conservative rule**: if either bin is unreliable, the pair falls back toward Gaussian.

The example below shows how different $\alpha$ values morph a correlation matrix from one method to the other.

In [ ]:
# --- Synthetic example: hybrid blend of two correlation matrices ---
np.random.seed(42)
n_bins = 12
energies = np.linspace(1, 6, n_bins)

# Gaussian parametric correlation (smooth exponential decay)
length_scale = 1.5
corr_gauss = np.exp(-0.5 * ((energies[:, None] - energies[None, :]) / length_scale)**2)

# Simulated KW correlation (data-driven, noisier, with some structure)
# Create a "realistic" KW correlation by adding structured noise
rng = np.random.default_rng(42)
raw = rng.normal(size=(200, n_bins))
# Add some block structure (experiments shared between nearby bins)
for i in range(n_bins):
    for j in range(max(0, i-2), min(n_bins, i+3)):
        raw[:, j] += 0.5 * raw[:, i]
corr_kw_raw = np.corrcoef(raw.T)
# Make it more distinct from Gaussian
corr_kw = 0.6 * corr_kw_raw + 0.4 * np.eye(n_bins)
np.fill_diagonal(corr_kw, 1.0)

# Simulate per-bin alpha: bins in the middle have more experiments (higher alpha)
# Edges have fewer experiments (lower alpha)
alpha_per_bin = np.array([0.1, 0.15, 0.4, 0.65, 0.8, 0.82, 0.78, 0.7, 0.5, 0.3, 0.1, 0.05])

# Build alpha_ij = min(alpha_i, alpha_j)
alpha_ij = np.minimum(alpha_per_bin[:, None], alpha_per_bin[None, :])

# Hybrid blend
corr_hybrid = alpha_ij * corr_kw + (1 - alpha_ij) * corr_gauss
np.fill_diagonal(corr_hybrid, 1.0)

# --- Plot ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

titles = ["Gaussian $\\rho_{\\text{Gauss}}$", "KW $\\rho_{\\text{KW}}$",
          "$\\alpha_i$ per bin", "Hybrid $\\rho_{\\text{hybrid}}$"]
matrices = [corr_gauss, corr_kw, None, corr_hybrid]

for ax, title, mat in zip(axes, titles, matrices):
    if mat is not None:
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal",
                       extent=[energies[0], energies[-1], energies[-1], energies[0]])
        ax.set_xlabel("Energy (MeV)", fontsize=10)
        ax.set_ylabel("Energy (MeV)", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
    else:
        ax.bar(energies, alpha_per_bin, width=0.35, color="C0", edgecolor="white")
        ax.set_xlabel("Energy (MeV)", fontsize=10)
        ax.set_ylabel("$\\alpha_i$", fontsize=10)
        ax.set_ylim(0, 1)
        ax.axhline(0.05, color="grey", ls="--", lw=0.8)
        ax.axhline(0.85, color="grey", ls=":", lw=0.8)
        ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=11)

plt.suptitle("Hybrid blend: $\\alpha$ controls how much KW structure is trusted per bin",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Step 5: End-to-end worked example

Let's trace the full pipeline for a single energy bin with concrete numbers.

In [ ]:
# --- End-to-end worked example for one energy bin ---

# ENDF bin: [4.8, 5.2] MeV with energy resolution σ_E = 0.15 MeV
bin_low, bin_high = 4.8, 5.2
sigma_E = 0.15   # bin's energy resolution (from TOF parameters)
print(f"ENDF bin: [{bin_low}, {bin_high}] MeV,  σ_E = {sigma_E} MeV")
print("=" * 65)

# Five experiments, each identified by entry.subentry.
# Each measured at a specific energy E_k.
# Note: σ_E is the bin's resolution, NOT the experiment's.
# If an experiment measured at multiple energies, only the closest to the bin is kept.
exp_data = [
    {"name": "Exp A (12345.002)", "E": 5.00, "n_angles": 15},
    {"name": "Exp B (23456.003)", "E": 4.95, "n_angles": 8},
    {"name": "Exp C (34567.004)", "E": 5.30, "n_angles": 42},
    {"name": "Exp D (45678.002)", "E": 5.80, "n_angles": 20},
    {"name": "Exp E (56789.005)", "E": 4.00, "n_angles": 12},
]

# Step 1: Compute overlap weights (same σ_E for all — it's the bin's property)
print("\n--- Step 1: Overlap weights (using the bin's σ_E for all experiments) ---")
weights = []
for exp in exp_data:
    w = norm.cdf(bin_high, exp["E"], sigma_E) - norm.cdf(bin_low, exp["E"], sigma_E)
    weights.append(w)
    print(f"  {exp['name']:25s}  E={exp['E']:.2f} MeV  ({exp['n_angles']:2d} angles, irrelevant here)  →  w = {w:.4f}")

weights = np.array(weights)

# Filter out negligible weights (min_weight = 1e-3 in the code)
min_weight = 1e-3
mask = weights >= min_weight
print(f"\n  Filtering w >= {min_weight}: keeping {mask.sum()} of {len(weights)} experiments")
exp_kept = [e for e, m in zip(exp_data, mask) if m]
weights_kept = weights[mask]
print(f"  Kept: {[e['name'] for e in exp_kept]}")

# Step 2: Compute N_eff_kw from surviving experiments
print("\n--- Step 2: N_eff,kw (Kish's ESS) ---")
w_frac = weights_kept / weights_kept.sum()
n_eff_kw = 1.0 / np.sum(w_frac**2)
for e, f in zip(exp_kept, w_frac):
    print(f"  {e['name']:25s}  f = {f:.4f}")
print(f"  N_eff,kw = 1 / Σ(f_k²) = 1 / {np.sum(w_frac**2):.4f} = {n_eff_kw:.2f}")

# Step 3: Other diagnostics
n_experiments = len(weights_kept)
f_max = float(np.max(w_frac))
print(f"\n--- Step 3: Additional diagnostics ---")
print(f"  n_experiments_kw: {n_experiments}")
print(f"  f_max (dominant fraction): {f_max:.4f}")

# Step 4: Compute alpha
print("\n--- Step 4: Compute alpha ---")
alpha_base_val = sigmoid_alpha(n_eff_kw)
print(f"  Base sigmoid(N_eff={n_eff_kw:.2f}): α_base = {alpha_base_val:.3f}")

alpha_final = full_alpha(n_eff_kw, n_experiments, f_max)
print(f"  After penalties: α_final = {alpha_final:.3f}")

# Check penalties
if n_experiments < 2:
    penalty_factor = 0.25 + 0.25 * n_experiments
    print(f"  ⚠ Experiment count penalty applied (n_exp={n_experiments} < 2): ×{penalty_factor:.2f}")
if f_max > 0.40:
    dom_penalty = 1.0 - 0.5 * (f_max - 0.40) / (1.0 - 0.40)
    print(f"  ⚠ Dominance penalty applied (f_max={f_max:.3f} > 0.40): ×{dom_penalty:.2f}")

# Interpretation
print(f"\n--- Interpretation ---")
print(f"  This bin uses {alpha_final:.0%} KW correlations + {1-alpha_final:.0%} Gaussian correlations.")
if alpha_final > 0.6:
    print(f"  → Mostly KW: enough experiments overlap → data-driven correlations are reliable.")
elif alpha_final > 0.3:
    print(f"  → Balanced blend: moderate experiment coverage.")
else:
    print(f"  → Mostly Gaussian: few experiments or one dominates → KW not reliable.")

## Summary

| Concept | What it is | Formula | Where in code |
|---------|-----------|---------|---------------|
| **Overlap weight** $w_k$ | Probability that experiment $k$'s energy falls in the ENDF bin, using the **bin's** $\sigma_E$ | $\Phi\left(\frac{E_{\text{high}} - E_k}{\sigma_E}\right) - \Phi\left(\frac{E_{\text{low}} - E_k}{\sigma_E}\right)$ | `compute_overlap_weight()` in `exfor_utils.py` |
| **Deduplication** | Per experiment (`entry.subentry`), only the energy with the highest $w_k$ is kept | `best_per_exp[exp_id] = max(w)` | `precompute_overlap_weights()` in `exfor_utils.py` |
| **$N_{\text{eff,kw}}$** | Effective number of experiments overlapping the bin (Kish's ESS) | $1 / \sum_k f_k^2$ where $f_k = w_k / \sum w_k$ | `compute_kw_diagnostics()` in `exfor_utils.py` |
| **$\alpha_i$** | Per-bin blend weight: how much to trust KW vs Gaussian correlations | Sigmoid on $N_{\text{eff,kw}}$ + experiment count penalty + dominance penalty | `compute_kw_reliability_alpha()` in `exfor_utils.py` |
| **$\alpha_{ij}$** | Pairwise blend weight for bin pair $(i,j)$ | $\min(\alpha_i, \alpha_j)$ | `exfor_to_endf_sampling_v2.py:2199` |
| **$\rho_{\text{hybrid}}$** | Final correlation used for Cholesky sampling | $\alpha_{ij} \rho_{\text{KW}} + (1-\alpha_{ij}) \rho_{\text{Gauss}}$ | `exfor_to_endf_sampling_v2.py:2202` |

**Important clarification:** $\sigma_E$ is the **bin's energy resolution** (from TOF parameters, stored in `bin_info.sigma_E_mev`). It is the same for all experiments at a given bin. The number of angular data points in an experiment does not affect the overlap weight — only the experiment's measurement energy relative to the bin boundaries matters.

**The point-level $N_{\text{eff}}$** (`compute_n_eff()` in `resample_AD.py`) is a separate quantity that measures how many data points effectively contribute to the Legendre fit at each bin. It uses kernel weights $\times$ $1/\sigma^2$ as combined weights. This is **only used as a diagnostic warning** (threshold=5.0) during the nominal fit — it does not affect any calculation.

**Note:** `compute_bin_reliability_alpha()` in `exfor_utils.py` exists but is **never called** in the current pipeline. Only the experiment-level `compute_kw_reliability_alpha()` is used.